<a href="https://colab.research.google.com/github/Fish210/3470-Competition-Team-Optimization-Model/blob/main/alliance_optimization_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
##### Imports #####
!pip install catboost
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

##### Load in Input Data #####
PATH = "/content/FTC_ALLIANCE_ML_MODEL_TRAINING_LT2026(team_match_actions).csv" # replace with file name
df = pd.read_csv(PATH)
df = df[df["team"].notna()].copy()

count_cols=[
    "auto_leave", "auto_artifact_cl_count", "auto_artifact_overflow_count", "auto_motif_match_count",
    "tele_artifact_cl_count", "tele_artifact_overflow_count","tele_motif_match_count", "tele_depot_count"
]
for c in count_cols:
  df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
##### Attatch scores to actions #####
df["auto_points"] = (
    3*df["auto_leave"]
    + 3*df["auto_artifact_cl_count"]
    + 1*df["auto_artifact_overflow_count"]
    + 2*df["auto_motif_match_count"]
)
df["tele_points"] = (
    3*df["tele_artifact_cl_count"]
    + 1*df["tele_artifact_overflow_count"]
    + 2*df["tele_motif_match_count"]
    + 1*df["tele_depot_count"]
)

df["total_points"] = df["auto_points"] + df["tele_points"]
df["dead_match"] = (df["total_points"] == 0).astype(int)

##### Baselines #####
team_stats = df.groupby("team").agg(
    mean_points=("total_points", "mean"),
    std_points=("total_points", "std"),
    dead_rate=("dead_match", "mean"),
    matches=("total_points", "count")
).reset_index()
team_stats["std_points"] = team_stats["std_points"].fillna(0)

team_stats["final_score"] = (
    team_stats["mean_points"]
    - 0.6*team_stats["std_points"]
    - 15*team_stats["dead_rate"] # Changed 'dead rate' to 'dead_rate'
)
team_stats = team_stats.sort_values("final_score", ascending = False)

print("=== Baseline ranking (FinalScore) ===")
print(team_stats[["team", "mean_points", "std_points", "dead_rate", "final_score", "matches"]].head(15)) #number of teams

##### Model A: Expected Points (CatBoost) #####
X = df[count_cols]
y = df["total_points"]

X_train, X_test, y_train, y_test, = train_test_split(
    X, y, test_size=0.25, random_state=42
)

model = CatBoostRegressor(
    depth=6,
    learning_rate=0.1,
    iterations=600,
    loss_function="MAE",
    verbose=False,
    random_seed=42
)
model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
print(f"\nCatBoost MAE on held-out matches: {mae:.2f} points")

# Predict expected points per team (average predicted value)
df["pred_points"] = model.predict(X)

pred_team = df.groupby("team").agg(
    pred_mean=("pred_points", "mean"),
    pred_std=("pred_points", "std"),
).reset_index()

pred_team["pred_std"] = pred_team["pred_std"].fillna(0)

# Combine with risk penalties (if any- exact same formula as pred_points)
pred_team = pred_team.merge(team_stats[["team", "dead_rate"]], on="team", how="left")
pred_team["model_final_score"] = pred_team["pred_mean"] - 0.6*pred_team["pred_std"] - 15*pred_team["dead_rate"]
pred_team = pred_team.sort_values("model_final_score", ascending=False)

##### Best partners for your FTC Team #####
TEAM = 3470 # your team
t3470 = pred_team[pred_team["team"] == TEAM]
if len(t3470) == 1:
  base = float(t3470["pred_mean"].iloc[0])
  partners = pred_team[pred_team["team"] != TEAM].copy()
  partners["expected_alliance_points"]= base + partners["pred_mean"]
  partners = partners.sort_values("expected_alliance_points", ascending=False)
  print(f"\n=== Best partners ffor {TEAM} by expected alliance points ===")
  print(partners[["team", "pred_mean", "expected_alliance_points", "dead_rate"]].head(10))
else:
  print("\nCouldn't find team in the data-check the team column.")

=== Baseline ranking (FinalScore) ===
       team  mean_points  std_points  dead_rate  final_score  matches
12  24689.0         58.2    4.549725        0.0    55.470165        5
5   18134.0         49.0   10.606602        0.0    42.636039        5
3   16278.0         52.8   20.535335        0.0    40.478799        5
9   21483.0         33.6    3.781534        0.0    31.331080        5
0    3470.0         34.8   16.558985        0.0    24.864609        5
13  26567.0         43.6   35.161058        0.0    22.503365        5
1    4998.0         27.2    9.679876        0.0    21.392074        5
7   21402.0         18.2    2.774887        0.0    16.535068        5
11  24610.0         15.4    5.639149        0.0    12.016511        5
8   21419.0         41.6   39.329378        0.4    12.002373        5
6   21027.0         14.8    9.093954        0.0     9.343628        5
14  31659.0         11.2    3.701351        0.0     8.979189        5
10  22047.0          3.2    2.167948        0.2    -